# Critical Input DEQN: Natural-Rate-Adjusted Taylor Rule

This notebook trains the natural-rate-adjusted Taylor-rule DEQN network using a natural benchmark network.  If a natural checkpoint exists it is loaded; otherwise the notebook trains the auxiliary natural network first.  The numerical natural oracle is deliberately not used inside BA training, because BA needs the natural rate and the oracle is too expensive in the rule-training loop.

In [ ]:
# Configure paths and natural-rate-adjusted Taylor settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError('No existing checkpoint found: ' + ', '.join(map(str, paths)))

NATURAL_CANDIDATES = [
    ARTIFACT_ROOT / 'natural' / 'checkpoints' / 'natural_best.pt',
    ARTIFACT_ROOT / 'natural' / 'natural.pt',
]
NATURAL_CKPT = next((path for path in NATURAL_CANDIDATES if path.exists()), None)
OUT = ARTIFACT_ROOT / 'modified_taylor'
OUT.mkdir(parents=True, exist_ok=True)

NATURAL_STEPS = 3_000
RULE_STEPS = 8_000
QMC_TRAIN = 256
QMC_VAL = 512
N_VAL_STATES = 1024
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
LOG_EVERY = 100
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
TARGET_RMS = None
TARGET_MAX_ABS = None
TARGET_SCENARIO_Q_RMS = 1e-2
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 512
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
CALM_RESIDUAL_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 25
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
NATURAL_AUDIT_STATES = 512
NATURAL_AUDIT_ORACLE_NODES = 32
NATURAL_AUDIT_CHUNK_SIZE = 4096

print(ROOT)
print(OUT)
print('NATURAL_CKPT =', NATURAL_CKPT)

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
# Train the Taylor rule that uses the natural benchmark.
# For BA, use the learned natural benchmark network rather than the numerical oracle.
# The BA rule needs R_n in every residual evaluation; doing that with the oracle is prohibitively slow.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', 'ba',
    '--natural-benchmark', 'network',
    '--natural-steps', str(NATURAL_STEPS),
    '--rule-steps', str(RULE_STEPS),
    '--rule-trainer', 'episode',
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--rule-scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--rule-calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--rule-calm-residual-weight', str(CALM_RESIDUAL_WEIGHT),
    '--rule-scenario-burnin', str(SCENARIO_BURNIN),
    '--rule-scenario-horizon', str(SCENARIO_HORIZON),
    '--rule-scenario-loss-interval', str(SCENARIO_LOSS_INTERVAL),
    '--target-scenario-q-rms', str(TARGET_SCENARIO_Q_RMS),
]
if NATURAL_CKPT is not None:
    cmd += ['--natural-checkpoint', str(NATURAL_CKPT)]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
run_stream(cmd, cwd=ROOT)


In [ ]:
# Inspect out-of-sample residual diagnostics for adjusted Taylor.
with (OUT / 'ba_eval.json').open('r', encoding='utf-8') as fh:
    ba_eval = json.load(fh)
ba_eval


In [ ]:
# Audit the learned natural-rate surrogate against the numerical oracle.
# This is not used for training; it checks whether the cheap BA training benchmark is accurate enough.
import numpy as np
import pandas as pd

from src.critical_input_deqn.config import NATURAL_OUTPUT_NAMES, NetworkConfig, QMCConfig
from src.critical_input_deqn.experiments import params_from_metadata
from src.critical_input_deqn.natural_oracle import natural_oracle_outputs
from src.critical_input_deqn.qmc import make_qmc_nodes
from src.critical_input_deqn.sampling import natural_from_rule_states, sample_rule_states
from src.critical_input_deqn.train import load_checkpoint, make_natural_net
from src.critical_input_deqn.transforms import decode_natural_outputs

BA_NATURAL_CKPT = NATURAL_CKPT or next(
    (path for path in [OUT / 'natural.pt', OUT / 'checkpoints' / 'natural_best.pt'] if path.exists()),
    None,
)
if BA_NATURAL_CKPT is None:
    raise FileNotFoundError('No natural benchmark checkpoint found for BA oracle audit.')

with (OUT / 'run_config.json').open('r', encoding='utf-8') as fh:
    run_config = json.load(fh)
params = params_from_metadata({'config': run_config})
dtype_t = torch.float64 if str(DTYPE).replace('torch.', '') == 'float64' else torch.float32
net_cfg = NetworkConfig(hidden_width=HIDDEN_WIDTH, hidden_depth=HIDDEN_DEPTH)

natural_net = make_natural_net(net_cfg, device=DEVICE, dtype=dtype_t)
load_checkpoint(BA_NATURAL_CKPT, natural_net, map_location=DEVICE)
natural_net.eval()

audit_qmc = QMCConfig(n_train=NATURAL_AUDIT_ORACLE_NODES, seed=991)
audit_nodes = make_qmc_nodes(NATURAL_AUDIT_ORACLE_NODES, cfg=audit_qmc, device=DEVICE, dtype=dtype_t)
z_rule = sample_rule_states(NATURAL_AUDIT_STATES, params=params, device=DEVICE, dtype=dtype_t, seed=9901)
z_n = natural_from_rule_states(z_rule)

with torch.no_grad():
    network_out = decode_natural_outputs(natural_net(z_n), NATURAL_OUTPUT_NAMES, params=params)
    oracle_out, oracle_info = natural_oracle_outputs(
        z_n,
        audit_nodes,
        params=params,
        qmc_cfg=audit_qmc,
        chunk_size=NATURAL_AUDIT_CHUNK_SIZE,
    )

rows = []
series = {}
for name in ['C_n', 'Y_n', 'R_n_real']:
    net_v = network_out[name].detach().cpu().numpy().reshape(-1)
    ora_v = oracle_out[name].detach().cpu().numpy().reshape(-1)
    err = net_v - ora_v
    rel = err / np.maximum(np.abs(ora_v), 1e-12)
    rows.append({
        'variable': name,
        'network_mean': float(np.mean(net_v)),
        'oracle_mean': float(np.mean(ora_v)),
        'bias': float(np.mean(err)),
        'mae': float(np.mean(np.abs(err))),
        'rms': float(np.sqrt(np.mean(err ** 2))),
        'max_abs': float(np.max(np.abs(err))),
        'rel_rms': float(np.sqrt(np.mean(rel ** 2))),
        'rel_max_abs': float(np.max(np.abs(rel))),
    })
    series[f'{name}_network'] = net_v
    series[f'{name}_oracle'] = ora_v
    series[f'{name}_error'] = err

audit_df = pd.DataFrame(rows)
audit_csv = OUT / 'ba_natural_surrogate_oracle_audit.csv'
audit_json = OUT / 'ba_natural_surrogate_oracle_audit.json'
audit_npz = OUT / 'ba_natural_surrogate_oracle_audit_series.npz'
audit_df.to_csv(audit_csv, index=False)
with audit_json.open('w', encoding='utf-8') as fh:
    json.dump({
        'natural_checkpoint': str(BA_NATURAL_CKPT),
        'states': int(NATURAL_AUDIT_STATES),
        'oracle_nodes': int(NATURAL_AUDIT_ORACLE_NODES),
        'oracle_info': oracle_info,
        'metrics': rows,
    }, fh, indent=2)
np.savez(audit_npz, **series)

print('Natural checkpoint:', BA_NATURAL_CKPT)
print('Saved:', audit_csv)
print('Saved:', audit_json)
print('Saved:', audit_npz)
display(audit_df)


In [ ]:
# IRF mechanism diagnostics for ba Taylor.
from src.critical_input_deqn.notebook_diagnostics import rule_ir_mechanism_diagnostics

BA_NATURAL_CKPT = NATURAL_CKPT or next(
    (path for path in [OUT / 'natural.pt', OUT / 'checkpoints' / 'natural_best.pt'] if path.exists()),
    None,
)
if BA_NATURAL_CKPT is None:
    raise FileNotFoundError('No natural benchmark checkpoint found for BA diagnostics.')

ba_ir_labels, ba_ir_defs, ba_mechanism_table = rule_ir_mechanism_diagnostics(
    artifact_root=ARTIFACT_ROOT,
    output_dir=OUT,
    policy='ba',
    natural_checkpoint=BA_NATURAL_CKPT,
    device=DEVICE,
    dtype=DTYPE,
    hidden_width=HIDDEN_WIDTH,
    hidden_depth=HIDDEN_DEPTH,
    natural_benchmark='network',
)
